# Day 8 - LLM foundations & API integration practical tasks
***

These tasks cover calling OpenAI API's and prompting.

## Explore how different model parameters and different prompts influence the response.
#### Here are some guidelines and advice you should consider:
* You can add as many cells as you need for each task.
* Use Markdown cells for your conclusions.
* For tasks that do not specify a model, you may choose any of the two available models: `gpt-5.6-luna` / `gpt-5.4-mini`.
* Within each comparison task, change only the model, prompt, or parameter specified in the instructions. Keep everything else - including the model, prompt, and other settings - the same across calls so you can compare the results fairly. For example, when testing different `temperature` values, use the same model, prompt, and output token limit (`max_output_tokens` for the Responses API or `max_completion_tokens` for the Chat Completions API).
* You can use both - [Responses API and Chat Completions API](https://medium.com/@sanathgoutham/openai-responses-api-vs-chat-completions-what-actually-changed-eeee86aff216) - to complete these tasks, but some model parameters are not supported by Responses API.
* LLMs understand [Markdown](https://www.markdownguide.org/cheat-sheet/) formatting in prompts. For example, wrapping text in double asterisks `(**important text**)` marks it as **bold** and can help signal emphasis.
* To copy text from a Markdown cell, open it in edit mode by clicking the pencil icon or double-clicking the cell.
* Optionally, paste API responses into a Markdown cell for better readability.
* Be creative and experiment.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ.pop("OPENAI_API_KEY", None)
load_dotenv()

True

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in the W2D3 .env file before running this cell.")

client = OpenAI(base_url="https://ds-ai-internship.openai.azure.com/openai/v1/",
                api_key=OPENAI_API_KEY)

#### 1. Write a Python script to call the OpenAI API and generate a response to the prompt below.

Prompt: `"What is artificial intelligence?"`

Use model `gpt-5.6-luna`.

In [4]:
prompt="What is artificial intelligence?"

response=client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{
        "role":"user", "content":prompt
    }],
)
print(response.choices[0].message.content)

Artificial intelligence (AI) is the field of creating computer systems that can perform tasks commonly associated with human intelligence, such as:

- Understanding language
- Recognizing images or speech
- Learning from data
- Solving problems
- Making predictions or decisions
- Generating text, images, music, or code

AI systems can be designed with explicit rules, trained using examples through machine learning, or built with a combination of both. Modern AI tools—such as chatbots, recommendation systems, and self-driving features—usually rely on machine-learning models that identify patterns in large amounts of data.


#### 2. Call the OpenAI API with different models

Prompt `"Explain the concept of machine learning to a 10-year-old."`

`model`: `gpt-5.6-luna` and `gpt-5.4-mini`.

Compare the outputs from each model. 

You can evaluate the differences in:
* Clarity of explanation.
* Simplicity of language (suitable for a 10-year-old).
* Creativity in the response.
* Response time.

In [8]:
import time

prompt="Explain the concept of machine learning to a 10-year-old."
models=["gpt-5.6-luna","gpt-5.4-mini"]

result={}
for model in models:
    start = time.perf_counter()
    response=client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":prompt}],
    )
    elapsed=time.perf_counter() -start
    result[model]={
        "text":response.choices[0].message.content,
        "time":elapsed,
        "tokens":response.usage.completion_tokens,
    }

    for model,r in result.items():
        print("="*70)
        print(f"MODEL: {model}")
        print(f"response time: {r['time']:.2f}s | output tokens : {r['tokens']}")
        print("-"*70)
        print(r["text"])
        print()

MODEL: gpt-5.6-luna
response time: 3.02s | output tokens : 232
----------------------------------------------------------------------
Machine learning is a way of teaching computers to recognize patterns and make guesses—without telling them every single rule.

Imagine showing a computer lots of pictures of cats and dogs. Each time, you tell it, “This is a cat” or “This is a dog.” The computer looks for patterns, such as ears, fur, size, and shape. After seeing many examples, it can look at a new picture and make a guess about whether it shows a cat or a dog.

It’s a bit like practicing:

- **Examples:** The computer studies lots of information.
- **Practice:** It finds patterns in that information.
- **Prediction:** It uses the patterns to make a guess about something new.

Machine learning is used in things like:

- Voice assistants understanding what you say
- Video apps recommending shows
- Email programs spotting spam
- Cars recognizing roads and objects

Computers don’t “think” o

* Clarity of explanation. Both use the cat and dog example, making it easy to follow, but luna is more structured, listing real world uses.
* Simplicity of language (suitable for a 10-year-old). gpt-mini uses shorter sentences and plainer wording then luna
* Creativity in the response.Both use the same examples ,but luna gives more real world uses
* Response time.gpt-mini is faster 2.10s vs 3.02s and uses less tokens.

#### 3. Call the OpenAI API with different `temperature` values

Prompt: `"Write a two sentence story about a robot learning to cook."`

`temperature` values: `0`, `0.7` and `1.5`. Generate three responses at each temperature.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare the outputs.

Tip: With Chat Completions API you can use parameter `n=3` to generate three different responses to your prompt with one call.

In [11]:
prompt= "Write a two sentence story about a robot learning to cook."
temperature=[0,0.7,1.5]

for temp in temperature:
    response= client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role":"user","content":prompt}],
        temperature=temp,
        reasoning_effort="none",
        n=3,
    )
    print("="*70)
    print(f"temp:{temp}")
    print("="*70)
    for i,choice in enumerate(response.choices, start=1):
        print(f"response{i}: {choice.message.content}\n")

temp:0
response1: The robot carefully studied recipes, measuring flour and stirring soup with precise, whirring motions until the kitchen smelled like warm bread. When it finally served dinner, it proudly announced that it had learned the secret ingredient was not code, but patience.

response2: The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, practiced motions. When it finally served dinner, the robot’s sensors glowed with pride as everyone agreed that its first homemade soup was delicious.

response3: The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, practiced motions. When it finally served dinner, the robot’s sensors glowed with pride as everyone agreed that its first homemade soup was delicious.

temp:0.7
response1: The robot carefully read its recipe book, stirring the soup with metallic precision and adding a pinch of sa

With higher temperature the chosen words are more varied/creative

#### 4. Call the OpenAI API with different `max_output_tokens`/`max_completion_tokens` values.

Prompt: `"Explain the concept of machine learning in simple terms."`

`max_output_tokens` (Response API call)/`max_completion_tokens` (Chat Completions API call) values: `50`, `100` and `200`.

Use `gpt-5.4-mini` for this task.

Compare the length and detail of the responses.

In [16]:

prompt="Explain the concept of machine learning in simple terms."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    max_completion_tokens=50
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Machine learning is a way of teaching computers to learn from data instead of being explicitly programmed for every task.

### Simple idea
Instead of writing rules like:
- “If the email has these words, mark it as spam”

you give
finish reason: length
CompletionUsage(completion_tokens=50, prompt_tokens=16, total_tokens=66, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 4, 'engine_ttft_ms': 26, 'engine_ttlt_ms': 229, 'pre_inference_ms': 114, 'service_tbt_ms': 4, 'service_ttft_ms': 240, 'service_ttlt_ms': 428, 'user_visible_ttft_ms': 126})


In [17]:

prompt="Explain the concept of machine learning in simple terms."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    max_completion_tokens=100
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Machine learning is a way for computers to **learn from examples** instead of being explicitly programmed for every task.

### Simple idea
Instead of writing exact rules like:
- “If an email contains these words, mark it as spam,”

you give the computer lots of examples:
- emails labeled **spam**
- emails labeled **not spam**

Then it finds patterns on its own and uses those patterns to make predictions on new emails.

### Everyday analogy
It’s a bit like teaching
finish reason: length
CompletionUsage(completion_tokens=100, prompt_tokens=16, total_tokens=116, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 4, 'engine_ttft_ms': 24, 'engine_ttlt_ms': 413, 'pre_inference_ms': 82, 'service_tbt_ms': 4, '

In [18]:

prompt="Explain the concept of machine learning in simple terms."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    max_completion_tokens=200
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Machine learning is a way for computers to **learn patterns from data** instead of being told every rule explicitly.

### Simple idea
Imagine teaching a computer to recognize cats:
- Instead of writing rules like “if it has whiskers, pointy ears, and fur, then it’s a cat,”
- you show it **many pictures of cats and non-cats**.
- Over time, it learns what features are usually associated with cats.

### In everyday terms
It’s like how humans learn from experience:
- A child learns what a dog is by seeing many dogs.
- A machine learns by looking at lots of examples.

### Common uses
Machine learning is used in:
- spam email filters
- product recommendations
- voice assistants
- image and face recognition
- predicting trends or outcomes

### The key idea
Instead of programming every step, you **train a model on data**, and the model then makes predictions or decisions based on what it learned.

If
finish reason: length
CompletionUsage(completion_tokens=200, prompt_tokens=16, total_tokens=21

With each increase of alloted tokens the sentance got filled out continuing the idea.

#### 5. Call the OpenAI API with different `top_p` values.

Prompt: `"Write a list of 5 completely random and unusual hobbies that no one has ever heard of."`

`top_p` values: `0.3`, `0.7`, and `1.0`.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare response variation as you adjust `top_p`.

In [19]:

prompt="Write a list of 5 completely random and unusual hobbies that no one has ever heard of."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    top_p=0.3,
    reasoning_effort="none"
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and sketching the shapes that clouds cast on the ground at different times of day.  
2. **Vintage receipt collecting** — hunting for old receipts and studying the prices, handwriting, and store designs from past decades.  
3. **Pebble balancing in rain** — arranging tiny stone towers specifically during light rain to see how water changes the balance.  
4. **Doorway acoustics testing** — standing in different doorframes and clapping to compare how each one changes sound.  
5. **Alphabetizing smells** — organizing objects, spices, or places by the first letter of the scent they remind you of.

If you want, I can make them even weirder or turn them into fake “serious” hobby descriptions.
finish reason: stop
CompletionUsage(completion_tokens=173, prompt_tokens=25, total_tokens=198, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0,

In [20]:

prompt="Write a list of 5 completely random and unusual hobbies that no one has ever heard of."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    top_p=0.7,
    reasoning_effort="none"
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Here are 5 completely random and unusual hobby ideas:

1. **Cloud Shadow Mapping** — tracking and sketching the shapes of shadows cast by clouds at different times of day.  
2. **Button Orchestra Conducting** — arranging buttons by size and material, then “conducting” imaginary symphonies for them.  
3. **Rust Pattern Photography** — photographing the abstract art formed by rust on old metal surfaces.  
4. **Silent Elevator Etiquette Study** — observing and documenting how people behave in elevators without speaking.  
5. **Pebble Biography Writing** — inventing detailed life stories for stones collected from different places.
finish reason: stop
CompletionUsage(completion_tokens=133, prompt_tokens=25, total_tokens=158, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image

In [21]:

prompt="Write a list of 5 completely random and unusual hobbies that no one has ever heard of."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    top_p=1,
    reasoning_effort="none"
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Here are 5 made-up, unusual hobby ideas:

1. **Cloud Shadow Collecting** — spending time tracking the shapes of shadows cast by clouds and “cataloging” them like rare specimens.  
2. **Typewriter Whispering** — sitting with old typewriters and trying to “encourage” them to produce smoother sounds and rhythms.  
3. **Pebble Remainder Sorting** — collecting tiny stones left behind after rain and organizing them by imagined personalities.  
4. **Lamp Post Listening** — standing near street lamps at night and noting the different hums, clicks, and buzzes each one makes.  
5. **Paper Breeze Folding** — folding sheets of paper into shapes specifically designed to react interestingly to indoor air currents.

If you want, I can make them sound even more bizarre, more realistic, or turn them into a fake hobby catalog.
finish reason: stop
CompletionUsage(completion_tokens=181, prompt_tokens=25, total_tokens=206, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audi

With low top_p the response is straight forward /focused with the decsriptions of weird hobbies, but with increased top_p the descriptions are more whimsical.

#### 6. Call the OpenAI API with different `frequency_penalty` values.

Prompt: `"Write a list of funny one-liner jokes about cats."`

`frequency_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the repetition of joke formats changes as you increase the `frequency_penalty`.

In [22]:

prompt= "Write a list of funny one-liner jokes about cats."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    reasoning_effort="none",
    frequency_penalty=0
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Sure — here are some funny one-liner cat jokes:

1. My cat isn’t lazy, he’s just on purr-mission impossible.  
2. Cats don’t have owners; they have staff.  
3. My cat likes to scratch the furniture because he believes in interior claw-design.  
4. I asked my cat for emotional support, and he ignored me with confidence.  
5. The cat joined the band because he had great purr-cussion.  
6. My cat’s favorite movie is The Silence of the Yams.  
7. I wanted a pet with commitment issues, so I got a cat.  
8. Cats are excellent at multitasking: they can annoy you and look adorable at the same time.  
9. My cat doesn’t run the house — he simply allows us to live in it.  
10. I bought my cat a bed, but he still prefers the box, because apparently he’s a minimalist.  
11. My cat is very spiritual; he practices meowditation.  
12. Cats are proof that “if I fits, I sits” is a legitimate lifestyle.  
13. My cat has a degree in paws-itive passive aggression.  
14. I tried to train my cat, but he said

In [23]:

prompt= "Write a list of funny one-liner jokes about cats."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    reasoning_effort="none",
    frequency_penalty=0.5
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Absolutely — here are some funny cat one-liners:

1. My cat isn’t spoiled; I’m just well-trained.
2. Cats: because punching things off tables is a personality.
3. I asked my cat for emotional support and got judged instead.
4. My cat likes to act like she pays rent.
5. The cat’s favorite hobby is sleeping in places I need.
6. If cats could talk, they wouldn’t.
7. My cat doesn’t do tricks — he does negotiations.
8. Cats are the reason the word “my” sounds so possessive.
9. I wanted a pet, but apparently I adopted a tiny furry landlord.
10. My cat’s favorite exercise is jumping to conclusions.
11. A cat will stare at you like you’re the weird one in your own house.
12. My cat only comes when called… if I’m opening food.
13. Cats: making every box look like a luxury apartment since forever.
14. My cat has seven lives and zero responsibilities.
15. I trust my cat to wake me up, mainly because she hates my sleep schedule.

If you want, I can also make them:
- more sarcastic
- more absurd
- 

In [24]:

prompt= "Write a list of funny one-liner jokes about cats."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    reasoning_effort="none",
    frequency_penalty=1
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Sure — here are some funny cat one-liners:

1. My cat doesn’t have an ego; she has a full-time public relations team.
2. I asked my cat for emotional support, and she walked away mid-sentence.
3. Cats don’t ignore you — they just prefer to respond in their own time zone.
4. My cat’s favorite exercise is jumping to conclusions.
5. A cat’s idea of helping is sitting directly on whatever you’re trying to do.
6. My cat thinks “personal space” is just a suggestion written by dogs.
7. I bought my cat a bed, but apparently the laundry basket was more luxurious.
8. Cats are great at self-care: 20 hours of sleep, 4 hours of judging humans.
9. My cat heard me say “time flies” and immediately attacked the clock.
10. If cats could text back, they still wouldn’t.

If you want, I can also make them:
- more sarcastic
- kid-friendly
- extra punny
- darker/more absurd
finish reason: stop
CompletionUsage(completion_tokens=217, prompt_tokens=18, total_tokens=235, completion_tokens_details=CompletionToken

With increased penalty the number of jokes reduced from 20 to 10. What i noticed is that it didnt penalize the word cat at all, it still is every joke.

#### 7. Call the OpenAI API with different `presence_penalty` values.

Prompt: `"Suggest ten names for a new line of eco-friendly cleaning products."`

`presence_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the diversity of generated names changes as you increase the `presence_penalty`.

In [25]:

prompt= "Suggest ten names for a new line of eco-friendly cleaning products."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    reasoning_effort="none",
    presence_penalty=0
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Here are ten name ideas for an eco-friendly cleaning product line:

1. **PureSprout**
2. **GreenGlint**
3. **EarthEase**
4. **EcoShine**
5. **FreshRoot**
6. **CleanBloom**
7. **Naturally Neat**
8. **VerdeVibe**
9. **BrightLeaf**
10. **GreenNest**

If you want, I can also suggest names that feel more **premium**, **playful**, or **scientific/modern**.
finish reason: stop
CompletionUsage(completion_tokens=110, prompt_tokens=19, total_tokens=129, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 4, 'engine_ttft_ms': 30, 'engine_ttlt_ms': 515, 'pre_inference_ms': 115, 'service_tbt_ms': 4, 'service_ttft_ms': 244, 'service_ttlt_ms': 724, 'user_visible_ttft_ms': 130})


In [26]:

prompt= "Suggest ten names for a new line of eco-friendly cleaning products."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    reasoning_effort="none",
    presence_penalty=0.5
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Here are ten name ideas for an eco-friendly cleaning product line:

1. **GreenGleam**
2. **EarthSpark**
3. **PurePulse**
4. **EcoShine**
5. **FreshSprout**
6. **VerdantClean**
7. **CleanKind**
8. **Leaf & Luster**
9. **NatureNest Clean**
10. **BrightBiome**

If you want, I can also suggest names that sound more:
- **premium/luxury**
- **minimalist/modern**
- **fun and playful**
- **all-natural/organic**
finish reason: stop
CompletionUsage(completion_tokens=124, prompt_tokens=19, total_tokens=143, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 11, 'engine_ttft_ms': 30, 'engine_ttlt_ms': 1385, 'pre_inference_ms': 208, 'service_tbt_ms': 11, 'service_ttft_ms': 366, 'service_ttlt_ms': 1718, 'user_visibl

In [27]:

prompt= "Suggest ten names for a new line of eco-friendly cleaning products."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{
        "role":"user", "content":prompt
    }],
    reasoning_effort="none",
    presence_penalty=1
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Here are ten name ideas for an eco-friendly cleaning product line:

1. **GreenGlow Clean**
2. **PurePlanet**
3. **EcoSpark**
4. **FreshLeaf Home**
5. **EarthKind Cleaning**
6. **NatureNook**
7. **CleanCurrent**
8. **Verdant Shine**
9. **Leaf & Lather**
10. **BlueSprout Clean**

If you want, I can also suggest names with a more premium, playful, modern, or minimalist vibe.
finish reason: stop
CompletionUsage(completion_tokens=107, prompt_tokens=19, total_tokens=126, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 11, 'engine_ttft_ms': 29, 'engine_ttlt_ms': 1223, 'pre_inference_ms': 71, 'service_tbt_ms': 11, 'service_ttft_ms': 176, 'service_ttlt_ms': 1365, 'user_visible_ttft_ms': 104})


the effect is hard to notice, but it is supposed to penalize if a word has been generated, and only product with green in its name disapeared , however "leaf" words are still there.

#### 8. Use the prompt "*Explain the importance of data preprocessing in machine learning.*" with the following system prompts:
1. "You are a university professor teaching a class of beginners."
2. "You are a technical expert explaining to a team of engineers."
3. "You are a comedian explaining this concept in a funny way."

Observe how the tone and style of the response change based on the system prompt.

In [29]:

prompt= "Explain the importance of data preprocessing in machine learning."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role" :"system","content":"You are a university professor teaching a class of beginners."},
        {
        "role":"user", "content":prompt
    }],
   
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Data preprocessing is one of the most important steps in machine learning because raw data is usually messy, inconsistent, and not directly useful for training a model.

## Why it matters

### 1. Improves data quality
Real-world data often has:
- missing values
- duplicates
- errors
- outliers
- inconsistent formats

Preprocessing helps clean these problems so the model learns from better data.

### 2. Makes data usable by algorithms
Machine learning models typically require numbers in a specific format.  
Preprocessing can:
- convert text into numerical features
- normalize or scale values
- encode categories like “red,” “blue,” “green”

Without this step, many algorithms cannot work properly.

### 3. Improves model performance
A model trained on well-preprocessed data usually:
- learns patterns more effectively
- makes more accurate predictions
- generalizes better to new data

### 4. Reduces noise
Not all data is useful. Some parts may be random or irrelevant.  
Preprocessing can re

In [30]:

prompt= "Explain the importance of data preprocessing in machine learning."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role" :"system","content":"You are a technical expert explaining to a team of engineers."},
        {
        "role":"user", "content":prompt
    }],
   
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Data preprocessing is one of the most important steps in machine learning because model quality depends heavily on the quality and consistency of the input data.

### Why it matters

1. **Improves model accuracy**
   - Real-world data is often noisy, incomplete, inconsistent, or irrelevant.
   - Cleaning and transforming it helps the model learn meaningful patterns instead of artifacts.

2. **Handles missing and bad data**
   - Missing values, duplicates, and outliers can distort training.
   - Preprocessing methods like imputation, filtering, and outlier treatment reduce this risk.

3. **Makes features usable**
   - Many algorithms work best when inputs are on a similar scale.
   - Techniques like normalization and standardization help models converge faster and perform better.

4. **Encodes categorical data**
   - Machine learning models typically require numerical inputs.
   - Preprocessing converts categories into numerical form using methods like one-hot encoding or label encoding

In [31]:

prompt= "Explain the importance of data preprocessing in machine learning."

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role" :"system","content":"You are a comedian explaining this concept in a funny way."},
        {
        "role":"user", "content":prompt
    }],
   
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Data preprocessing is the **“get your life together before the model shows up”** step of machine learning.

Imagine trying to teach a student using:

- notes written in 5 languages,
- some pages upside down,
- missing homework,
- and one sheet that is just a drawing of a potato.

That’s raw data. Machine learning models are smart, but they’re not magical. If you feed them messy data, they’ll learn messy patterns and then confidently make bad predictions. Very “I have no idea what I’m doing, but I said it with confidence.”

### Why preprocessing matters

**1. It improves data quality**  
Real-world data is often full of:
- missing values,
- duplicates,
- errors,
- noise,
- weird outliers,
- and columns that make no sense.

Preprocessing cleans that up so the model doesn’t get distracted by garbage.

**2. It helps the model learn better**  
If one feature is “age” and another is “annual income,” the income numbers may be much larger. Some models get dramatic about this and overfocus on t

Proffesor is freindly,with simple analogies and definitions.
Technical expert is concise and precise
Comedian makes jokes and playful analogies, but when it gets technical it gives up the persona.

#### 9. Write a Zero-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!".

Use it in your API call.

In [32]:

review = "The product is amazing, and I love using it every day!"
zero_shot_prompt=f"""Analyze the sentiment of the following product review 
Review: "{review}"
"""

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":zero_shot_prompt
    }],
   
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

The sentiment of the review is **very positive**.  
Words like **“amazing”** and **“love”** indicate strong satisfaction and enthusiasm.
finish reason: stop
CompletionUsage(completion_tokens=35, prompt_tokens=31, total_tokens=66, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 4, 'engine_ttft_ms': 33, 'engine_ttlt_ms': 159, 'pre_inference_ms': 92, 'service_tbt_ms': 3, 'service_ttft_ms': 226, 'service_ttlt_ms': 337, 'user_visible_ttft_ms': 133})


#### 10. Write a Few-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!"

Use it in your API call.

Observe how the model's performance changes and compare the response to Zero-Shot output.

In [34]:

review = "The product is amazing, and I love using it every day!"
few_shot_prompt=f"""Analyze the sentiment of the product review. Answer with one word: Positive,Negative or Neutral.

Review: "the car died after 2 days"
sentiment:negtive
Review: "Great quality, realy fast"
sentiment:positive
Review: "Works as described, nothing special"
sentiment:neutral


Review: "{review}"
"""

response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":few_shot_prompt
    }],
   
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

Positive
finish reason: stop
CompletionUsage(completion_tokens=4, prompt_tokens=89, total_tokens=93, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 2, 'engine_ttft_ms': 35, 'engine_ttlt_ms': 46, 'pre_inference_ms': 103, 'service_tbt_ms': 8, 'service_ttft_ms': 254, 'service_ttlt_ms': 285, 'user_visible_ttft_ms': 151})


Zero-shot gives a longer free form answer with explanation.
few-shot gives a single word answer, as the example.

#### 11. Optimize input tokens.

Prompt: `"As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."`

Rewrite the prompt to use as few input tokens as possible while preserving its meaning and the requirement to answer in two simple sentences. The response should remain relevant, clear, and accurate.

Run the original and revised prompts using the same model and settings. Compare their input token counts and assess whether shortening the prompt affected the response quality.

You can use the API response’s usage information to check token counts.

In [36]:

prompt= "in two simple sentences, explain why a recomendation system shows a romantic comedy to a user who loves sci-fi movies."
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    reasoning_effort="none",
   
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

A recommendation system might show a romantic comedy because it notices some users who enjoy sci-fi also like movies with similar themes, actors, or humor. It uses patterns from many viewers’ choices, so it suggests something that seems likely to appeal even if it’s a different genre.
finish reason: stop
CompletionUsage(completion_tokens=58, prompt_tokens=30, total_tokens=88, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 5, 'engine_ttft_ms': 34, 'engine_ttlt_ms': 355, 'pre_inference_ms': 87, 'service_tbt_ms': 5, 'service_ttft_ms': 206, 'service_ttlt_ms': 522, 'user_visible_ttft_ms': 119})


In [37]:

prompt= "As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    reasoning_effort="none",
   
)
print(response.choices[0].message.content)
print("finish reason:",response.choices[0].finish_reason)
print(response.usage)

The recommendation system may have noticed that many people who like sci-fi also enjoy that romantic comedy, so it thinks you might like it too. It could also be trying to mix in something different based on your viewing history or a recent change in your taste.
finish reason: stop
CompletionUsage(completion_tokens=54, prompt_tokens=94, total_tokens=148, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 5, 'engine_ttft_ms': 31, 'engine_ttlt_ms': 306, 'pre_inference_ms': 166, 'service_tbt_ms': 5, 'service_ttft_ms': 314, 'service_ttlt_ms': 586, 'user_visible_ttft_ms': 149})


Both outputs give the same sentiment even with shortened prompt

#### 12. Optimize output tokens.

Prompt: `"Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"`

Rewrite the prompt to request a concise, plausible explanation in 10 output tokens or fewer.

Run the original and revised prompts using the same model and settings. Compare their output token counts and check whether the shorter response still provides a meaningful explanation. Refine your prompt if necessary.

You can use the API response’s usage information to check token counts.

In [46]:

prompt= " in 5 words or fewer: Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Because it spotted a crossover.
9


In [47]:

prompt= " Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

A few common reasons:

- **One off-viewing can skew it:** If you recently watched, liked, searched for, or even paused on a romantic comedy, the app may treat that as a strong signal.
- **Shared profiles:** If someone else uses the same account, their tastes can affect your recommendations.
- **Similar audience overlap:** The app may think sci-fi fans also enjoy certain romantic comedies based on broader viewing patterns.
- **Popularity/trending boost:** Some apps recommend widely popular titles even if they’re not a perfect fit for your history.
- **Metadata mix-up:** A movie with sci-fi elements or a cross-genre title may get recommended because the system sees partial overlap.
- **Cold start/limited data:** If it doesn’t have enough data on you, it may guess more broadly.

If you want, I can also explain how recommendation systems decide this and how to “train” your app to show more sci-fi.
195


The OG prompt produced 195 output tokens, a long explanation with several reasons.
The modified promt produced 9 output tokens, and response being one of the reasons the OG prompt gave. 

#### 13. Expand the given prompt accordingly to achieve the requested output from the list below.

Prompt: `"Explain the importance of exercise." `

Expected output (from three separate prompts):
1. One sentence long
2. Paragraph format
3. Bullet points

In [48]:

prompt= "Explain the importance of exercise in one sentance" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Exercise is important because it keeps your body healthy, strengthens your heart and muscles, and improves your mood and energy.
26


In [49]:

prompt= "Explain the importance of exercise in a single paragraph of 4 to 6 sentences." 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Exercise is important because it helps keep the body strong, healthy, and full of energy. Regular physical activity improves heart health, builds muscle and bone strength, and helps maintain a healthy weight. It also supports mental health by reducing stress, improving mood, and increasing focus. In addition, exercise can lower the risk of many diseases, such as diabetes and high blood pressure, making it an essential part of a healthy lifestyle.
87


In [50]:

prompt= "Explain the importance of exercise as a bulleted list of 5 bullet points. Only use bullet points" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

- Improves cardiovascular health by strengthening the heart and reducing the risk of heart disease.
- Helps maintain a healthy weight by burning calories and supporting metabolism.
- Boosts mental health by reducing stress, anxiety, and symptoms of depression.
- Increases strength, flexibility, and endurance, making daily activities easier.
- Supports overall well-being by improving sleep, energy levels, and immune function.
80


- Improves cardiovascular health by strengthening the heart and reducing the risk of heart disease.
- Helps maintain a healthy weight by burning calories and supporting metabolism.
- Boosts mental health by reducing stress, anxiety, and symptoms of depression.
- Increases strength, flexibility, and endurance, making daily activities easier.
- Supports overall well-being by improving sleep, energy levels, and immune function.

#### 14. Expand this prompt to get more creative responses.

Prompt: `"Explain Artificial Intelligence."`

Expected output (from three separate prompts):
1. Metaphor
2. Story
3. Funny

In [52]:

prompt= "Explain Artificial Intelligence using a vivid methaphor.Build the explanation around a single comparison, and keep it short" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Artificial Intelligence is like a **giant library with an invisible librarian**.

You bring it a question, and instead of pointing to one book, the librarian has read millions of them, spotted patterns, and quickly hands you the most useful answer. It doesn’t “think” like a person—it **predicts what fits best** based on everything it has learned.
76


In [53]:

prompt= "Explain Artificial Intelligence using a short story" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Maya worked in a busy library where hundreds of books needed sorting every day. One afternoon, the library got a new helper named **AI**.

At first, Maya thought AI was a robot with arms and wheels. But AI was more like a very smart pattern-finder. It watched how books were arranged, noticed which ones were borrowed most often, and learned to suggest where each book should go.

When a child asked for “a funny book about space,” AI quickly searched thousands of titles and found the best match. When the library got noisy, AI helped predict the busiest hours so staff could prepare. It never got tired, and it kept learning from every new task.

Maya smiled and said, “So AI is a machine that learns from data and helps us make better decisions.”

AI beeped softly, as if to say, “Exactly.”

And from that day on, the library ran faster, the readers found books more easily, and Maya learned that Artificial Intelligence is not magic—it’s technology that can learn, recognize patterns, and assist 

In [54]:

prompt= "Explain Artificial Intelligence in a funny way, like a comedian" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Artificial Intelligence, or AI, is basically when we teach computers to act like they have a brain… but without the brain drama.

It’s like giving a toaster a PhD and saying, “Congratulations, you’re now in charge of predicting what humans want.”

AI learns by looking at tons of data, which is kind of like a student who never studies the textbook, just binge-watches every possible exam question until it starts guessing confidently. That’s why AI can be amazing at things like recognizing faces, translating languages, or recommending exactly the same song you’ve already heard 400 times. Very helpful. Very creepy. Very “how did you know that, computer?”

Think of AI as a super-enthusiastic intern:
- It never sleeps
- It works ridiculously fast
- It makes weird mistakes with total confidence
- And every so often, it says something so wrong you stare at it like, “Who let you near the internet?”

There are different kinds of AI too. Some are simple and just follow rules, like a vending machi

#### 15. Expand this prompt for different audiences.

Prompt: `"Explain climate change."`

Expected audience (from three separate prompts):
1. Kids
2. General public
3. Politicians

In [55]:

prompt= "Explain climate change to a 8 year old child, use simple words and friendly tone" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Climate change means the Earth is getting a little warmer over time.

Think of the Earth like a big house. It needs to stay at a good temperature for people, animals, and plants. But some gases in the air trap extra heat from the Sun, like a blanket that is too thick. This makes the Earth warmer than it should be.

Why does this happen?
- We burn fuels like coal, oil, and gas for cars, factories, and electricity.
- That puts extra gases into the air.
- These gases hold in heat.

What can happen?
- Some places can get hotter.
- Weather can become stronger, with bigger storms or long dry times.
- Ice in very cold places can melt.
- Animals and plants may find it harder to live in some places.

What can we do?
- Save electricity by turning off lights.
- Walk, bike, or use public transport when possible.
- Recycle and reuse things.
- Plant trees.
- Be kind to the Earth every day.

So, climate change is when our planet slowly gets warmer, and we need to help keep Earth healthy for everyone.

In [56]:

prompt= "Explain climate change to the general public, for adults and teen with no science background. Be clear and use plain language" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Climate change means the Earth’s climate is changing over time, and it is getting warmer because people are adding too much heat-trapping gas to the air.

## The simple idea

The Earth is surrounded by an atmosphere, like a blanket.

Some gases in the air, such as carbon dioxide and methane, trap heat from the sun. This is natural and important, because without it Earth would be too cold for most life.

But since the Industrial Revolution, humans have been burning huge amounts of coal, oil, and gas for energy, transport, factories, and heating. That has added extra heat-trapping gases to the atmosphere. As a result, more heat stays in the Earth system.

## What is happening because of this?

The main effect is that the planet is warming, but it does not only mean “hotter weather.” It also means changes in weather patterns and more extreme events.

Examples include:

- hotter summers and more heatwaves
- heavier rain in some places, causing floods
- longer dry periods in some places, ca

In [57]:

prompt= "Explain climate change to a politicatians. Focus on economy and infrastructure" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Here’s a politician-friendly way to explain climate change, focused on **economy and infrastructure**:

---

## Climate change is not just an environmental issue — it is a cost issue

Climate change affects the economy in three major ways:

### 1. It damages existing infrastructure
- Roads buckle in extreme heat.
- Bridges, rail lines, and airports are stressed by flooding, storms, and sea-level rise.
- Power grids fail more often during heat waves and storms.
- Water systems face droughts, contamination, and saltwater intrusion.

**Bottom line:** We pay more to repair and replace assets sooner than expected.

### 2. It increases operating costs
- Businesses spend more on cooling, insurance, and backup systems.
- Farmers face lower yields and higher irrigation costs.
- Supply chains are disrupted by storms, fires, and floods.
- Insurance premiums rise, and in some places coverage becomes unavailable.

**Bottom line:** Climate risk raises the cost of doing business.

### 3. It slows eco

#### 16. Expand this prompt accordingly to instructions below.

Prompt: `"Describe a sunset."`

Expected result (from three separate prompts):

1. Add more details to the prompt, e.g., on what details should it focus on.
2. Constrain the prompt by forbidding it to use words 'sun' and 'light'.
3. Change the output in the style of Shakespeare.

In [58]:

prompt= "Describe a sunset, focus on the golden color and the horizon" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

The sun lowers toward the horizon, spilling a deep golden light across the sky. The color seems to soften everything it touches—clouds, water, distant fields—until the whole world is wrapped in warm amber. Along the horizon, the glow gathers in a bright band, where gold melts into orange and then into a fading blue above. It feels calm and endless, as if the day is slowly breathing out its last warm light.
90


In [59]:

prompt= "Describe a sunset, but dont use words like **sun** and **light**" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

The sky blazed with bands of amber, rose, and deep violet, slowly melting into one another as day loosened its grip. Clouds caught fire along their edges, then softened into dusky silk. The horizon turned molten, as if the world were cooling from within. Shadows lengthened across the land, and the air grew still, holding that brief, breathless moment before night settled in.
83


In [60]:

prompt= "Describe a sunset in the style of William Shakespeare, use Early modern English" 
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Lo, as the day in weary splendour dies,  
The sun doth sink beneath the western brim,  
And gild the clouds with gold that softly flies,  
Whilst twilight, like a gentle veil, grows dim.  

The skies do blush with crimson, deep and fair,  
As though the heavens had taken fire and peace;  
The murm’ring wind doth kiss the balmy air,  
And all the world is hush’d in sweet surcease.  

The hills do wear a purple mantle now,  
The streams reflect a pale and dying flame;  
Upon the earth doth dim the shadow bow,  
And night draw on, yet still in beauty came.  

So ends the day with soft and solemn grace,  
And leaves his rosy footprint in the place.
163


#### 17. Write your observations from tasks 13-16, e.g., how different prompts influence complexity, tone, style and focus of the responses. 

Task 13:Asking for one sentence, a paragraph or bulletin points, changed the structure and lenght of the answers, but content was consistent.
Task 14; Naming a creative form changed the style and tone dramaticaly.
task15:Giving an audience changed the complexity, vocabulary and focus realy smoothly.
task 16: Giving details or giving constraints did not stop the description from being unclear what it was about. Also I have not read the original works of Shakespeare, but the description seams accurate enough.

#### 18. Generate a Python code with API call.

Write a prompt that generates a Python function to calculate the factorial of a number. Once you have written the prompt, use it to call the API and generate the code. Test the generated code to ensure it calculates the factorial correctly for various inputs.

In [ ]:
code_prompt=("write a Python function called factorial(n) that calculates the factorial of non negative integer n.Raise error for negative numbers.")
 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":code_prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

```python
def factorial(n):
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if n < 0:
        raise ValueError("n must be a non-negative integer")

    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

Example:
```python
print(factorial(5))  # 120
print(factorial(0))  # 1
```
109


In [63]:
def factorial(n):
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if n < 0:
        raise ValueError("n must be a non-negative integer")

    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


print(factorial(5))  # 120
print(factorial(0))
print(factorial(-4))

120
1


ValueError: n must be a non-negative integer

#### 19. Provide the following buggy Python code to the API and ask it to fix the errors:

```
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
```

Check if the fixed code is working.

In [64]:
code_prompt=("fix this python code :def add_numbers(a, b) return a + b print(add_numbers(5, 10)")

 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":code_prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Here’s the fixed Python code:

```python
def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))
```

What was fixed:
- Added the missing `:` after the function definition
- Put `print(...)` on a new line
- Closed the parentheses properly

It will output:

```python
15
```
79


In [65]:
def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))

15


#### 20. Summarize this text to 2-3 sentences by using API call.

"Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."

[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai)

In [67]:
code_prompt=("Summerize this text to 2-3 sentences :Generative AI, sometimes called gen AI, refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.Diffusion models, first seen in 2014, which add noise to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney.")

 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":code_prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

Generative AI uses deep learning models to create new content like text, images, video, or audio based on a user prompt. It works by learning patterns from training data and generating similar but original outputs, with major advances coming from VAEs, diffusion models, and transformers.
58


#### 21. Translate the following idioms into a language of your choice, ensuring the meaning is preserved (not just word-for-word translation):

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."

Try to improve the response as much as possible and evaluate the accuracy of the final output.

In [69]:
code_prompt= ("""Translate the idioms to Latvian :
 "Its raining cats and dogs."
 "Break a leg."
 "Kill two birds with one stone."
 "Spill the beans."
 "A blessing in disguise."""
)

 
response=client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {
        "role":"user", "content":code_prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

- **It’s raining cats and dogs.** — **Gāž kā ar spaiņiem.**
- **Break a leg.** — **Lai veicas!**
- **Kill two birds with one stone.** — **Ar vienu šāvienu nošaut divus zaķus.**
- **Spill the beans.** — **Izpļāpāt noslēpumu.**
- **A blessing in disguise.** — **Laime nelaimē.**
413


The accuracy of the translation is astonishing, I did not expect the accuracy to be so spot on.

#### 22. Generate a paragraph about a cat without using the letter 'e'/'E'.

Write a prompt that forces the AI to produce a three-sentence story about a cat while strictly forbidding it to use letter 'e'.

Use `gpt-5.4-mini` for this task.

Explain why your prompt is effective.

In [71]:
code_prompt= ("write three-sentence story about a cat while strictly not using letter **e**, check every word you write and make sure it has no **e**."
)

 
response=client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {
        "role":"user", "content":code_prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

A sly cat sat on a warm mat.  
In dim moonlight, it ran and had fun.  
Joy did fill all who saw it.
39


I tried emphasize the e whith a ** on both sides, also i repeated the instruction a second time.

#### 23. Extract JSON Data from the input text in two different ways.

Input Text: "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

Extract the information from the text above using two different methods:

1. Write a prompt asking the AI to return a JSON object with the fields `destination`, `start_date`, and `stay_duration`.

2. Instead of putting rules in the prompt, provide a specific JSON schema that defines the field names and their required data types `destination` (the first city), `start_date` (formatted as YYYY-MM-DD), and `stay_duration` (the number of nights as an integer).

Which method yields a more reliable result for a computer program to process, and why?

In [75]:

text="Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

prompt= f""" return a JSON object with the fields destination, start_date, and stay_duration from the text below.
Text: "{text}"
"""


 
response=client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {
        "role":"user", "content":prompt
    }],
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

{"destination":"Tokyo","start_date":"June 12","stay_duration":"5 nights"}
55


In [77]:
schema= {
    "name":"trip_info",
    "strict":True,
    "schema":{
        "type":"object",
        "properties":{
            "destination":{"type":"string","description":"city of the trip"},
            "start_date":{"type":"string","description":"Date as YYYY-MM-DD"},
            "stay_duration":{"type":"integer","description":"Number of nights"},

        },
        "required":["destination","start_date","stay_duration"],
        "additionalProperties":False,

    },
}
text="Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."


response=client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {
        "role":"user", "content":f'Extract the trip info from text:"{text}"' }],
        response_format={"type":"json_schema","json_schema":schema},
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

{"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5}
91


The JSON schema method is more reliable. With a schema, the API enforces the field names, required fields, data type and format.

#### 24. Create a structured table comparing different programming languages.

Write a prompt that generates a table in this format (example below) with accurate information.

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |
| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |
| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |

**Tip**: To enhance readability, you can copy and paste the generated table into a Markdown cell.

In [81]:
prompt="""Create a markdown table comparing programming languages.
Columns in this order: Language|Use case|Advantages|Disadvantages
rows: Python, java, javascript.
fill out the table with accurate information.
Keep it short. 4-5 words """


response=client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {
        "role":"user", "content":prompt }],
        
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

| Language | Use case | Advantages | Disadvantages |
|---|---|---|---|
| Python | Data science, automation, web | Readable syntax, huge ecosystem | Slower execution, higher memory |
| java | Enterprise systems, Android applications | Portable, scalable, strongly typed | Verbose syntax, slower development |
| javascript | Interactive websites, server-side applications | Runs everywhere, flexible ecosystem | Dynamic typing, tooling fragmentation |
305


| Language | Use case | Advantages | Disadvantages |
|---|---|---|---|
| Python | Data science, automation, web | Readable syntax, huge ecosystem | Slower execution, higher memory |
| java | Enterprise systems, Android applications | Portable, scalable, strongly typed | Verbose syntax, slower development |
| javascript | Interactive websites, server-side applications | Runs everywhere, flexible ecosystem | Dynamic typing, tooling fragmentation |

#### 25. Generate a quiz about machine learning basics.

Look at the example below and write a prompt that generates a quiz with identical formatting. Ensure the title formatting, list markers (a, b, c, d), and the correct answer indicator (*) are accurately replicated.

**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *

In [86]:
prompt="""Generate a quiz about machine learning basics. Follow the format EXACTLY, including title formatting, list markers (a, b, c, d), and the correct answer indicator (*) 
**Machine learning basics quiz**

1.question
a)
b)
c)
d)correct * 

2.question
a)correct *
b)
c)
d) 

Exatcly one option per question is correct, and it ends with "*".
"""

 

response=client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {
        "role":"user", "content":prompt }],
        
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

**Machine learning basics quiz**

1.What is machine learning?
a)A method for computers to learn patterns from data *
b)A type of computer hardware
c)A programming language
d)A database system

2.Which type of learning uses labeled data?
a)Unsupervised learning
b)Reinforcement learning
c)Supervised learning *
d)Random learning

3.What is a feature in machine learning?
a)A data input used by a model *
b)The final prediction
c)A programming error
d)A computer monitor

4.Which task predicts a continuous numerical value?
a)Classification
b)Clustering
c)Regression *
d)Sorting

5.What is overfitting?
a)When a model performs well on all new data
b)When a model memorizes training data and performs poorly on new data *
c)When a model has too little data to train
d)When a model is not given any features

6.Which algorithm is commonly used for classification?
a)Decision tree *
b)Linear search
c)File compression
d)Text formatting

7.What is the purpose of a training set?
a)To evaluate the final mod

#### 26. Create a comprehensive online course outline for beginners.

Write a prompt that generates entire course outline, following the example provided below. Ensure the prompt is detailed enough to cover all sections, with clear formatting and practical guidance. 

Sections to include in course outline:
- Course overview - description, objectives, target audience.
- Module breakdown - 5 modules with title, short description, three key learning outcomes and an interactive assignment in the end of each module.
- Assessment plan - a table outlining the assessment methods with description and weighting.
- Certification criteria - requirements to earn a certificate.
- Marketing strategy - step-by-step guide for promoting the course.

Include a detailed description of the type of information each section should contain. Maintain the **same formatting**, **section titles**, and **course name**. While the generated descriptions, modules and marketing strategy steps may differ and incorporate new ideas and concepts, it must remain aligned with the theme of the task.

Put the final output of your API call in a Markdown cell. 

#### Example:
---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---

In [88]:
prompt = """Create a comprehensive online course outline for beginners on the topic: "Python Programming for Absolute Beginners".

Return the outline in Markdown, using EXACTLY the structure and formatting below. Replace the angle-bracket placeholders with real content.

FORMAT:

---

## Course Outline: <Course Name>

---

### Course Overview
<One paragraph of 3 to 5 sentences describing the course.>

#### Objectives
- <objective 1>
- <objective 2>
- <objective 3>

#### Target Audience
- <audience group 1>
- <audience group 2>
- <audience group 3>

---

### Module Breakdown

Write exactly 5 modules. Each module uses this layout, and a "---" line separates one module from the next:

#### Module <number>: <Module Title>
- **Description:** <one or two sentences>
- **Key Learning Outcomes:**
  1. <outcome 1>
  2. <outcome 2>
  3. <outcome 3>
- **Interactive Assignment:** <one practical task for the end of the module>

---

### Assessment Plan

A Markdown table with exactly these columns: | Assessment Method | Description | Weightage (%) |
Include 4 rows. The weightages must add up to 100%.

---

### Certification Criteria
Start with the line "To earn a certificate of completion, students must:" followed by a numbered list of 3 requirements.

---

### Marketing Strategy
A numbered, step-by-step guide of 5 steps for promoting the course. Each step has a bold title followed by one sentence.

RULES:
- Keep the section titles, heading levels, bold labels, numbering and "---" separators exactly as shown.
- Progress from basic to more advanced across the modules.
- The content must be practical, accurate and suitable for complete beginners.
- Return only the outline, with no introduction, no closing remarks and no code fences.
"""

 

response=client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {
        "role":"user", "content":prompt }],
        
    
   
)
print(response.choices[0].message.content)
print(response.usage.completion_tokens)

---

## Course Outline: Python Programming for Absolute Beginners

---

### Course Overview
This course introduces complete beginners to Python programming through clear explanations, guided practice, and practical projects. Students will learn how to write, run, and debug Python programs while building a strong foundation in programming concepts. The course progresses from basic syntax and data types to control flow, functions, data structures, and simple real-world applications. No previous programming experience is required.

#### Objectives
- Understand Python syntax, programming concepts, and development tools.
- Write programs using variables, data types, conditions, loops, functions, and collections.
- Apply problem-solving and debugging techniques to create practical Python projects.

#### Target Audience
- Complete beginners with no prior programming experience.
- Students, career changers, and professionals interested in learning Python.
- Anyone seeking a practical foundatio

---

## Course Outline: Python Programming for Absolute Beginners

---

### Course Overview
This course introduces complete beginners to Python programming through clear explanations, guided practice, and practical projects. Students will learn how to write, run, and debug Python programs while building a strong foundation in programming concepts. The course progresses from basic syntax and data types to control flow, functions, data structures, and simple real-world applications. No previous programming experience is required.

#### Objectives
- Understand Python syntax, programming concepts, and development tools.
- Write programs using variables, data types, conditions, loops, functions, and collections.
- Apply problem-solving and debugging techniques to create practical Python projects.

#### Target Audience
- Complete beginners with no prior programming experience.
- Students, career changers, and professionals interested in learning Python.
- Anyone seeking a practical foundation for further study in software development, automation, or data analysis.

---

### Module Breakdown

#### Module 1: Getting Started with Python
- **Description:** Students are introduced to Python, programming environments, and the basic structure of a Python program. They will learn how to install Python or use an online interpreter and write their first instructions.
- **Key Learning Outcomes:**
  1. Install Python and choose an appropriate code editor or online development environment.
  2. Use print statements, comments, indentation, and basic Python syntax.
  3. Identify and correct simple syntax and runtime errors.
- **Interactive Assignment:** Create a personal introduction program that displays the student’s name, interests, and a short motivational message.

---

#### Module 2: Variables, Data Types, and User Input
- **Description:** Students learn how programs store and process information using variables and common data types. They will also practice receiving input from users and converting values into appropriate types.
- **Key Learning Outcomes:**
  1. Create, update, and use variables with strings, integers, floats, and Boolean values.
  2. Perform arithmetic, comparison, and logical operations.
  3. Collect user input and convert it safely for use in calculations.
- **Interactive Assignment:** Build a simple personal budget calculator that accepts income and expenses and displays the remaining balance.

---

#### Module 3: Making Decisions and Repeating Tasks
- **Description:** This module introduces control flow, enabling programs to make decisions and repeat actions. Students will use conditional statements and loops to solve increasingly practical problems.
- **Key Learning Outcomes:**
  1. Use `if`, `elif`, and `else` statements to control program behavior.
  2. Apply `for` and `while` loops to process repeated tasks.
  3. Use loop controls such as `break` and `continue` appropriately.
- **Interactive Assignment:** Develop a number-guessing game that provides feedback, tracks attempts, and ends when the correct number is guessed.

---

#### Module 4: Functions and Data Collections
- **Description:** Students learn how to organize reusable code with functions and manage groups of related values using lists, tuples, dictionaries, and sets. The module emphasizes writing readable and maintainable programs.
- **Key Learning Outcomes:**
  1. Define functions with parameters, return values, and meaningful names.
  2. Create, access, modify, and iterate through lists and dictionaries.
  3. Choose appropriate data structures for storing and organizing information.
- **Interactive Assignment:** Create a command-line contact manager that allows users to add, search for, update, and remove contacts.

---

#### Module 5: Files, Errors, and a Beginner Project
- **Description:** The final module introduces file handling, exception management, and basic program organization. Students will combine the skills from the course to build and present a small practical Python application.
- **Key Learning Outcomes:**
  1. Read from and write to text files using appropriate file-handling techniques.
  2. Handle common errors with `try` and `except` blocks.
  3. Plan, build, test, and explain a complete beginner-level Python project.
- **Interactive Assignment:** Build a personal task manager that stores tasks in a file, allows users to add and view tasks, and handles invalid input gracefully.

---

### Assessment Plan

| Assessment Method | Description | Weightage (%) |
|---|---|---:|
| Module Exercises | Short coding activities and practice tasks completed throughout Modules 1 and 2. | 20 |
| Control Flow Quiz and Game | Quiz covering decision-making and loops, combined with evaluation of the number-guessing game. | 20 |
| Data Management Assignment | Contact manager demonstrating functions, lists, dictionaries, and basic program organization. | 25 |
| Final Python Project | Practical task manager project assessed for functionality, code quality, error handling, and explanation. | 35 |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve an overall assessment score of at least 70%.
2. Complete all module assignments and submit the final Python project.
3. Demonstrate understanding of core Python concepts through quizzes, practical tasks, and project review.

---

### Marketing Strategy
1. **Define the Beginner Benefit:** Promote the course as a friendly, zero-experience introduction that helps learners build real Python programs from the first lesson.
2. **Showcase Practical Projects:** Feature examples such as the budget calculator, number-guessing game, contact manager, and task manager in promotional materials.
3. **Use Educational Content Marketing:** Share short Python tips, beginner mistakes, and mini coding challenges on social media, blogs, and email newsletters.
4. **Offer a Low-Commitment Entry Point:** Provide a free introductory lesson, sample exercise, or live workshop to let prospective students experience the teaching approach.
5. **Build Trust Through Learner Results:** Highlight student projects, testimonials, completion rates, and certificate benefits in course pages and promotional campaigns.


####